# PRCP-1002: Handwritten Digits Recognition — Improved Version (v2)

## Objective
Classify 28 × 28 grayscale handwritten digits into ten classes (0–9) with MNIST, compare KNN, RBF-SVM, MLP (ANN) and CNN **fairly**, pick the model with validation data only, and evaluate the selected model **once** on the official test set.

> **Before submitting:** use *Kernel → Restart & Run All*, wait for it to finish, then **save the notebook with its outputs**. The previous evaluation lost marks because no executed outputs were present. Every conclusion in this notebook is generated from the run's own numbers, so an executed copy contains the actual best model, metrics and recommendation.

## What changed compared with v1 (mapped to the evaluation feedback)
| Evaluation feedback | Change in this version |
|---|---|
| Outputs / numbers missing; conclusion did not name a best model | Final report and conclusion are **generated from the results** (selected model, hyper-parameters, validation + test metrics, confusions, speed). Run all cells and save with outputs. |
| EDA: quantitative outlier / stroke-density / class-specific analysis | Added stroke-density by class, IQR outlier analysis with example images, connected-component (broken-stroke) analysis, class-similarity heatmap, constant-pixel count. |
| Preprocessing: is `StandardScaler` after normalisation justified? | Added an **ablation** (normalise only vs normalise + standardise) per model; the winner is chosen on validation data and used for tuning. |
| CNN used the comparison validation set for early stopping | A separate **early-stopping set** is carved out of the training pool. The 6,000-image validation set is used only for comparison. The MLP early-stops on the same set. |
| Memory use and split class counts not reported | Per-class split table and memory table added. |
| Narrow hyper-parameter search; single CNN | Wider grids for KNN, SVM, MLP; three CNN variants (baseline, deeper + BatchNorm, augmentation). |
| No repeated validation / larger equal-size experiment | **Repeated random 15,000-image subsets** (mean ± std) and a **learning curve** over equal training sizes, kept separate from the primary ranking. |
| No per-class metrics, confidence analysis, single-image latency | Added per-class F1 heatmap, confidence/selective-prediction analysis, single-image latency (median, p95), and model size. |
| Differences between models not tested | Exact McNemar tests between models and 95 % confidence intervals. |

## 0. Setup and configuration

In [ ]:
import os, sys, time, copy, pickle, hashlib, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, scipy
from scipy import ndimage
from scipy.stats import binomtest, spearmanr
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             precision_recall_fscore_support, classification_report,
                             confusion_matrix)

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    HAS_TF = True
except ImportError:
    HAS_TF = False

SEED = 42
np.random.seed(SEED)
if HAS_TF:
    tf.keras.utils.set_random_seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Got `batch_size`")

# ---- configuration -------------------------------------------------------
SMOKE = os.environ.get("MNIST_SMOKE") == "1"   # internal quick test switch; leave unset
QUICK = False            # True = much smaller grids (a few minutes) for a dry run
RUN_FULL_TRAIN = False   # True = add the full training pool to the learning curve (slow for SVM/KNN)
if SMOKE:
    FIT_N, VAL_N, ES_N, SIZES, ROB_SEEDS = 600, 250, 150, [100, 200, 400], [11, 22]
else:
    FIT_N, VAL_N, ES_N, SIZES, ROB_SEEDS = 15000, 6000, 3000, [2500, 5000, 10000, 30000], [11, 22, 33]

print("Python:", sys.version.split()[0], "|", platform.platform())
print("NumPy:", np.__version__, "| Pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)
print("TensorFlow:", tf.__version__ if HAS_TF else "NOT INSTALLED (CNN will be skipped - pip install tensorflow)")
print("Seed:", SEED, "| fit subset:", FIT_N, "| validation:", VAL_N, "| early-stop set:", ES_N)

## 1. Load the data

In [ ]:
def load_mnist():
    if SMOKE:   # tiny stand-in used only to test the code path; never for reported results
        from sklearn.datasets import load_digits
        d = load_digits()
        imgs = np.stack([ndimage.zoom(a, 3.5, order=1) for a in d.images])
        imgs = np.clip(imgs * 255 / 16, 0, 255).astype(np.uint8)
        a, b, c, e = train_test_split(imgs, d.target, test_size=0.28, stratify=d.target, random_state=SEED)
        return (a, c), (b, e)
    try:
        from tensorflow.keras.datasets import mnist
        return mnist.load_data()
    except Exception as err:
        print("Keras MNIST loader unavailable (", type(err).__name__, ") - trying OpenML")
        from sklearn.datasets import fetch_openml
        X, y = fetch_openml("mnist_784", version=1, as_frame=False, return_X_y=True)
        X = X.reshape(-1, 28, 28).astype(np.uint8); y = y.astype(int)
        return (X[:60000], y[:60000]), (X[60000:], y[60000:])

(x_train, y_train), (x_test, y_test) = load_mnist()
print("Training images:", x_train.shape, "| labels:", y_train.shape)
print("Test images:    ", x_test.shape, "| labels:", y_test.shape)
print("dtype:", x_train.dtype, "| pixel range:", int(x_train.min()), "to", int(x_train.max()))
print("Labels:", np.unique(y_train))

# Task 1 — Exploratory Data Analysis (EDA)

In [ ]:
n_const = int((x_train.reshape(len(x_train), -1).std(axis=0) == 0).sum())
print("Missing values:", int(np.isnan(x_train.astype(np.float32)).sum()))
print("Mean / std of pixels: %.2f / %.2f" % (x_train.mean(), x_train.std()))
print("Zero-valued (background) pixels: %.2f%%" % (100 * (x_train == 0).mean()))
print("Always-constant pixel positions in the training set: %d of %d (carry no information)" % (n_const, 784))

**Reading:** the zero-pixel share is background, not missing data. Constant pixel positions can be removed without losing information, which is one reason dimensionality-sensitive models (KNN, SVM) benefit from scaling choices evaluated later.

In [ ]:
# Ten random examples per class, to show within-class handwriting variation (v1 showed only one).
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(10, 10, figsize=(8, 8))
for d in range(10):
    ids = rng.choice(np.where(y_train == d)[0], size=10, replace=False)
    for j, ax in enumerate(axes[d]):
        ax.imshow(x_train[ids[j]], cmap="gray"); ax.axis("off")
    axes[d, 0].set_title(f"digit {d}", fontsize=8, loc="left")
plt.suptitle("Ten random training examples per digit (rows = classes)")
plt.tight_layout(); plt.show()

In [ ]:
# Class balance: counts and percentages.
class_counts = pd.Series(y_train).value_counts().sort_index()
class_summary = pd.DataFrame({"Count": class_counts, "Percentage": 100 * class_counts / len(y_train)}).round(2)
display(class_summary)
imbalance_ratio = class_counts.max() / class_counts.min()
ax = class_counts.plot(kind="bar", figsize=(8, 3.6))
ax.set(title="Training images per digit", xlabel="Digit", ylabel="Images")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
print(f"Largest class: {class_counts.idxmax()} ({class_counts.max()}), smallest: {class_counts.idxmin()} ({class_counts.min()}), "
      f"ratio {imbalance_ratio:.2f} -> " + ("mildly imbalanced; macro metrics and stratified splits are used" if imbalance_ratio < 2 else "clearly imbalanced"))

In [ ]:
# Stroke density and brightness per class, with IQR outlier analysis.
ink = (x_train > 127).sum(axis=(1, 2))                 # number of stroke pixels
df_eda = pd.DataFrame({"label": y_train, "ink_px": ink, "mean_intensity": x_train.mean(axis=(1, 2))})
by_class = df_eda.groupby("label").agg(images=("ink_px", "size"), ink_mean=("ink_px", "mean"),
                                       ink_std=("ink_px", "std"), intensity_mean=("mean_intensity", "mean")).round(2)
display(by_class)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot([df_eda.loc[df_eda.label == d, "ink_px"] for d in range(10)])
axes[0].set_xticks(range(1, 11)); axes[0].set_xticklabels(range(10))
axes[0].set(title="Stroke pixels (>127) per image, by digit", xlabel="Digit", ylabel="stroke pixels")
axes[1].hist(df_eda.mean_intensity, bins=40, edgecolor="black")
axes[1].set(title="Mean intensity per image", xlabel="mean pixel intensity", ylabel="images")
plt.tight_layout(); plt.show()

def iqr_outlier(s):
    q1, q3 = s.quantile([.25, .75]); k = q3 - q1
    return (s < q1 - 1.5 * k) | (s > q3 + 1.5 * k)
df_eda["ink_outlier"] = df_eda.groupby("label")["ink_px"].transform(iqr_outlier)
df_eda["ink_z"] = df_eda.groupby("label")["ink_px"].transform(lambda s: (s - s.mean()) / (s.std() + 1e-9))
outlier_rate = df_eda.ink_outlier.mean()
print("IQR outliers in stroke density (within class): %.2f%% of images" % (100 * outlier_rate))
display(df_eda.groupby("label").ink_outlier.mean().mul(100).round(2).to_frame("outlier % by class").T)

extreme = df_eda.reindex(df_eda.ink_z.abs().sort_values(ascending=False).index).head(10)
fig, axes = plt.subplots(1, 10, figsize=(14, 2.4))
for ax, (i, r) in zip(axes, extreme.iterrows()):
    ax.imshow(x_train[i], cmap="gray"); ax.axis("off"); ax.set_title(f"{int(r.label)}  z={r.ink_z:.1f}", fontsize=8)
plt.suptitle("Most extreme stroke-density outliers (within their own class)"); plt.tight_layout(); plt.show()

In [ ]:
# Connected components: a clean digit is usually one connected stroke; several components flag broken or noisy writing.
struct8 = np.ones((3, 3), dtype=int)
cc_idx = np.concatenate([rng.choice(np.where(y_train == d)[0], size=min(500, int((y_train == d).sum())), replace=False) for d in range(10)])
cc = np.array([ndimage.label(x_train[i] > 127, structure=struct8)[1] for i in cc_idx])
cc_df = pd.DataFrame({"label": y_train[cc_idx], "components": np.clip(cc, 0, 3)})
cc_tab = pd.crosstab(cc_df.label, cc_df.components, normalize="index").mul(100).round(1)
cc_tab.columns = [f"{c}{'+' if c == 3 else ''} component(s) %" for c in cc_tab.columns]
display(cc_tab)
multi_share = float((cc_df.components >= 2).mean())
multi_top = (cc_df.assign(m=cc_df.components >= 2).groupby("label").m.mean().sort_values(ascending=False))
print("Share of sampled images with 2+ separate strokes: %.1f%%; highest for digits %s" % (100 * multi_share, list(multi_top.index[:3])))

In [ ]:
# Mean image per class, class-similarity heatmap and the pixel-variance map.
x_train_norm = x_train.astype("float32") / 255.0
means = np.stack([x_train_norm[y_train == d].mean(axis=0).ravel() for d in range(10)])
unit = means / np.linalg.norm(means, axis=1, keepdims=True)
sim = unit @ unit.T

fig, axes = plt.subplots(1, 12, figsize=(16, 2.4), gridspec_kw={"width_ratios": [1] * 10 + [0.1, 2.6]})
for d in range(10):
    axes[d].imshow(means[d].reshape(28, 28), cmap="gray"); axes[d].axis("off"); axes[d].set_title(str(d), fontsize=9)
axes[10].axis("off")
im = axes[11].imshow(x_train_norm.std(axis=0), cmap="magma"); axes[11].axis("off"); axes[11].set_title("pixel std map", fontsize=9)
plt.suptitle("Mean image by digit and where pixel variation lives"); plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 5)); plt.imshow(sim, cmap="viridis"); plt.colorbar(label="cosine similarity")
plt.xticks(range(10)); plt.yticks(range(10)); plt.title("Similarity of class-mean images")
for r in range(10):
    for c in range(10):
        plt.text(c, r, f"{sim[r, c]:.2f}", ha="center", va="center", fontsize=6, color="w")
plt.tight_layout(); plt.show()
pair_sim = sorted([((i, j), sim[i, j]) for i in range(10) for j in range(i + 1, 10)], key=lambda t: -t[1])
print("Most similar class-mean pairs (predicted confusion candidates):", [(p, round(float(s), 3)) for p, s in pair_sim[:5]])

In [ ]:
# Exact duplicates inside the training set and between training and test sets.
tr_hash = pd.Series([hashlib.md5(a.tobytes()).hexdigest() for a in x_train])
te_hash = set(hashlib.md5(a.tobytes()).hexdigest() for a in x_test)
dup_groups = tr_hash.value_counts(); dup_groups = dup_groups[dup_groups > 1]
leak = int(tr_hash.isin(te_hash).sum())
print("Exact-duplicate groups inside training set:", len(dup_groups))
print("Training images that also appear in the test set:", leak)
print("Note: near-duplicates and same-writer overlap are not detected.")

## EDA findings (generated from the numbers above)

In [ ]:
top_sim = pair_sim[0]
display(Markdown(f"""
- **Size/format:** {len(x_train):,} training and {len(x_test):,} test images, 28 × 28 `uint8`, 10 classes, no missing values; {100*(x_train==0).mean():.1f}% of pixels are background and {n_const} pixel positions are always blank.
- **Balance:** class counts range from {class_counts.min():,} to {class_counts.max():,} (ratio {imbalance_ratio:.2f}); the data are close to balanced, but macro-averaged metrics and stratified splits are still used.
- **Stroke density:** mean stroke pixels range from {by_class.ink_mean.min():.0f} (digit {by_class.ink_mean.idxmin()}) to {by_class.ink_mean.max():.0f} (digit {by_class.ink_mean.idxmax()}). {100*outlier_rate:.1f}% of images are within-class IQR outliers (very thin/thick writing), so models must cope with stroke-width variation.
- **Broken strokes:** {100*multi_share:.1f}% of sampled images contain two or more separate stroke components; this is highest for digits {list(multi_top.index[:3])}.
- **Similarity:** the most similar class-mean pair is {top_sim[0]} (cosine {top_sim[1]:.3f}); this is tested against real validation confusions later.
- **Leakage check:** {len(dup_groups)} exact-duplicate groups in training; {leak} training images appear in the test set.
- **Modelling implications:** scale pixels to [0, 1]; keep stratified splits; test whether extra standardisation helps; give the CNN augmentation (shifts/rotations) a chance because writing varies in thickness, slant and position.
"""))

# Task 2 — Preprocessing and experimental design
- Pixels → float32 in [0, 1]; flattened to 784 features for KNN/SVM/MLP; 28×28×1 tensors for the CNN.
- **Four disjoint sets:** (1) 6,000-image **validation** set — used only for comparing/selecting models; (2) 3,000-image **early-stopping** set — used only by the MLP and CNN to decide when to stop; (3) the **15,000-image fit subset** every model trains on; (4) the official **test** set — touched once, at the end.
- Whether extra `StandardScaler` helps is decided empirically (ablation below), not assumed.

In [ ]:
x_test_norm = x_test.astype("float32") / 255.0
X_all = x_train_norm.reshape(len(x_train_norm), -1)
X_test_flat = x_test_norm.reshape(len(x_test_norm), -1)
to_img = lambda A: A.reshape(-1, 28, 28, 1)

all_idx = np.arange(len(y_train))
rest_idx, val_idx = train_test_split(all_idx, test_size=VAL_N, random_state=SEED, stratify=y_train)
pool_idx, es_idx = train_test_split(rest_idx, test_size=ES_N, random_state=SEED, stratify=y_train[rest_idx])
sub_idx, _ = train_test_split(pool_idx, train_size=FIT_N, random_state=SEED, stratify=y_train[pool_idx])

X_fit, y_fit = X_all[sub_idx], y_train[sub_idx]
X_val, y_val = X_all[val_idx], y_train[val_idx]
X_es, y_es = X_all[es_idx], y_train[es_idx]
assert not (set(sub_idx) & set(val_idx)) and not (set(sub_idx) & set(es_idx)) and not (set(val_idx) & set(es_idx))
print("fit:", X_fit.shape, "| validation:", X_val.shape, "| early-stop:", X_es.shape, "| test:", X_test_flat.shape, "| pool available for scaling studies:", len(pool_idx))

split_table = pd.DataFrame({"Fit subset": pd.Series(y_fit).value_counts().sort_index(),
                            "Validation": pd.Series(y_val).value_counts().sort_index(),
                            "Early-stop": pd.Series(y_es).value_counts().sort_index(),
                            "Test": pd.Series(y_test).value_counts().sort_index()})
split_table.loc["Total"] = split_table.sum()
display(split_table)
mem = pd.DataFrame({"MB": {"x_train uint8 (raw)": x_train.nbytes / 1e6, "x_train float32 [0,1]": x_train_norm.nbytes / 1e6,
                           "fit subset float32 (flat)": X_fit.nbytes / 1e6, "validation float32 (flat)": X_val.nbytes / 1e6}}).round(1)
display(mem)

### Helper functions (fit, predict, score, latency, size)

In [ ]:
def score_predictions(y_true, y_pred):
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0)}

def base_gamma(Xf, scaling):
    """Equivalent of sklearn's gamma='scale' for the (optionally standardised) training data."""
    Z = StandardScaler().fit_transform(Xf) if scaling == "standard" else Xf
    return 1.0 / (Xf.shape[1] * Z.var())

def fit_mlp(Xf, yf, scaling, hidden, alpha, max_epochs=60, patience=6):
    """MLP trained epoch by epoch; stops on the separate early-stopping set (never on the validation set)."""
    scaler = StandardScaler().fit(Xf) if scaling == "standard" else None
    prep = (lambda A: scaler.transform(A)) if scaler is not None else (lambda A: A)
    Xa, Xe = prep(Xf), prep(X_es)
    clf = MLPClassifier(hidden_layer_sizes=hidden, alpha=alpha, batch_size=128, random_state=SEED)
    best, best_acc, wait = None, -1.0, 0
    for _ in range(max_epochs):
        clf.partial_fit(Xa, yf, classes=np.arange(10))
        acc = clf.score(Xe, y_es)
        if acc > best_acc + 1e-4:
            best, best_acc, wait = copy.deepcopy(clf), acc, 0
        else:
            wait += 1
            if wait >= patience: break
    return Pipeline([("scale", scaler if scaler is not None else "passthrough"), ("model", best)])

def build_cnn(kind):
    L = layers
    aug = [L.RandomRotation(0.06, fill_mode="constant", fill_value=0.0),
           L.RandomTranslation(0.08, 0.08, fill_mode="constant", fill_value=0.0)] if kind == "base_aug" else []
    if kind in ("base", "base_aug"):
        body = [L.Conv2D(32, 3, activation="relu"), L.MaxPooling2D(2), L.Conv2D(64, 3, activation="relu"), L.MaxPooling2D(2),
                L.Flatten(), L.Dense(128, activation="relu"), L.Dropout(0.30)]
    else:  # deeper_bn
        body = [L.Conv2D(32, 3, padding="same"), L.BatchNormalization(), L.Activation("relu"),
                L.Conv2D(32, 3, padding="same"), L.BatchNormalization(), L.Activation("relu"), L.MaxPooling2D(2), L.Dropout(0.25),
                L.Conv2D(64, 3, padding="same"), L.BatchNormalization(), L.Activation("relu"), L.MaxPooling2D(2), L.Dropout(0.25),
                L.Flatten(), L.Dense(128, activation="relu"), L.Dropout(0.40)]
    m = models.Sequential([L.Input(shape=(28, 28, 1))] + aug + body + [L.Dense(10, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

cnn_histories = {}
def train_cnn(kind, Xf, yf, epochs=25, patience=3):
    tf.keras.utils.set_random_seed(SEED)
    m = build_cnn(kind)
    stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)
    h = m.fit(to_img(Xf), yf, epochs=epochs, batch_size=128, validation_data=(to_img(X_es), y_es), callbacks=[stop], verbose=0)
    return m, h.history

def cfg_label(cfg):
    p, f = cfg["params"], cfg["family"]
    if f == "KNN": return f"k={p['k']}, weights={p['weights']}"
    if f == "SVM": return f"RBF, C={p['C']}, gamma={p['gamma_mult']}x scale"
    if f == "MLP": return f"hidden={p['hidden']}, alpha={p['alpha']}"
    return f"architecture={p['kind']}"

def fit_candidate(cfg, Xf, yf):
    """Fit one configuration on (Xf, yf); returns (fitted estimator, training seconds)."""
    fam, sc, p = cfg["family"], cfg["scaling"], cfg["params"]
    scaler = StandardScaler() if sc == "standard" else "passthrough"
    t0 = time.perf_counter()
    if fam == "KNN":
        est = Pipeline([("scale", scaler), ("model", KNeighborsClassifier(n_neighbors=p["k"], weights=p["weights"], n_jobs=-1))]).fit(Xf, yf)
    elif fam == "SVM":
        est = Pipeline([("scale", scaler), ("model", SVC(kernel="rbf", C=p["C"], gamma=p["gamma_mult"] * base_gamma(Xf, sc)))]).fit(Xf, yf)
    elif fam == "MLP":
        est = fit_mlp(Xf, yf, sc, p["hidden"], p["alpha"])
    else:
        est, hist = train_cnn(p["kind"], Xf, yf); cnn_histories[(len(yf), cfg_label(cfg))] = hist
    return est, time.perf_counter() - t0

def predict_any(fam, est, X):
    if fam == "CNN":
        return np.argmax(est.predict(to_img(X), verbose=0, batch_size=512), axis=1)
    return est.predict(X)

def timed_predict(fam, est, X):
    t0 = time.perf_counter(); pred = predict_any(fam, est, X); return pred, time.perf_counter() - t0

def confidence_any(fam, est, X):
    if fam == "CNN":
        return est.predict(to_img(X), verbose=0, batch_size=512).max(axis=1), "softmax probability"
    if hasattr(est, "predict_proba"):
        return est.predict_proba(X).max(axis=1), "predicted probability"
    return est.decision_function(X).max(axis=1), "SVM margin (uncalibrated)"

def single_latency_ms(fam, est, X, n=150):
    """Median / 95th-percentile time to classify ONE image, single-threaded for KNN."""
    if fam == "KNN":
        est = copy.deepcopy(est); est.set_params(model__n_jobs=1)
    if fam == "CNN":
        f = lambda r: int(np.argmax(est(to_img(r[None, :]), training=False).numpy(), axis=1)[0])
    else:
        f = lambda r: int(est.predict(r[None, :])[0])
    rows = X[:n]
    for r in rows[:5]: f(r)
    ts = []
    for r in rows:
        t0 = time.perf_counter(); f(r); ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts)), float(np.percentile(ts, 95))

def model_size_mb(fam, est):
    return est.count_params() * 4 / 1e6 if fam == "CNN" else len(pickle.dumps(est)) / 1e6

def df_to_md(df, floatfmt="{:.4f}"):
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in r) + " |")
    return "\n".join(lines)

FAMILIES = ["KNN", "SVM", "MLP"] + (["CNN"] if HAS_TF else [])
if not HAS_TF: print("CNN skipped: TensorFlow is not installed.")

### Preprocessing ablation: is `StandardScaler` on top of [0, 1] scaling worth it?
Each classical model is trained twice at a fixed default setting — normalisation only vs normalisation + standardisation — and compared on the validation set. The winner per model family is used in the tuning below.

In [ ]:
defaults = {"KNN": dict(k=5, weights="distance"), "SVM": dict(C=5, gamma_mult=1.0), "MLP": dict(hidden=(128, 64), alpha=1e-4)}
abl = []
for fam, p in defaults.items():
    for sc in ["none", "standard"]:
        cfg = dict(family=fam, scaling=sc, params=p)
        est, ts = fit_candidate(cfg, X_fit, y_fit)
        pred, tp = timed_predict(fam, est, X_val)
        abl.append(dict(Family=fam, Scaling="[0,1] only" if sc == "none" else "[0,1] + StandardScaler", scaling_key=sc,
                        Accuracy=accuracy_score(y_val, pred), MacroF1=f1_score(y_val, pred, average="macro"), TrainSec=ts, ValPredSec=tp))
abl = pd.DataFrame(abl)
display(abl.drop(columns="scaling_key").round(4))
best_scaling = {}
for fam in defaults:
    r = abl[abl.Family == fam].set_index("scaling_key")
    best_scaling[fam] = "standard" if r.loc["standard", "MacroF1"] > r.loc["none", "MacroF1"] + 1e-9 else "none"
best_scaling["CNN"] = "none"
print("Scaling chosen per family (validation macro-F1):", best_scaling)

# Task 3 — Model training and validation-based tuning
Every candidate trains on the **same 15,000 images** and is scored on the **same validation set**; the configuration with the highest validation macro-F1 is selected within each family (ties: the faster one).

In [ ]:
GRIDS = {
    "KNN": [dict(k=k, weights=w) for k in ([3, 5] if QUICK else [1, 3, 5, 7, 9]) for w in ["uniform", "distance"]],
    "SVM": [dict(C=C, gamma_mult=g) for C in ([5] if QUICK else [2, 5, 10]) for g in ([1.0] if QUICK else [0.5, 1.0, 2.0])],
    "MLP": [dict(hidden=h, alpha=a) for h in ([(128, 64)] if QUICK else [(128,), (256,), (128, 64), (256, 128)]) for a in ([1e-4] if QUICK else [1e-4, 1e-3])],
    "CNN": [dict(kind=k) for k in (["base"] if QUICK else ["base", "deeper_bn", "base_aug"])],
}
tuning_log, cand = [], {}
for fam in FAMILIES:
    for p in GRIDS[fam]:
        cfg = dict(family=fam, scaling=best_scaling[fam], params=p)
        est, ts = fit_candidate(cfg, X_fit, y_fit)
        pred, tp = timed_predict(fam, est, X_val)
        row = dict(Family=fam, Configuration=cfg_label(cfg), Scaling=cfg["scaling"], **score_predictions(y_val, pred),
                   TrainSec=ts, ValPredSec=tp)
        tuning_log.append(row); cand[(fam, row["Configuration"])] = (cfg, est, pred)
        print(f"{fam:4s} {row['Configuration']:42s} acc={row['Accuracy']:.4f}  macroF1={row['Macro F1']:.4f}  train={ts:6.1f}s")
tuning_df = pd.DataFrame(tuning_log)

In [ ]:
display(tuning_df.sort_values(["Family", "Macro F1"], ascending=[True, False]).round(4).reset_index(drop=True))

selected = {}
for fam in FAMILIES:
    best = tuning_df[tuning_df.Family == fam].sort_values(["Macro F1", "TrainSec"], ascending=[False, True]).iloc[0]
    cfg, est, pred = cand[(fam, best.Configuration)]
    selected[fam] = dict(cfg=cfg, est=est, pred=pred, row=best)

if HAS_TF:
    key = (len(y_fit), cfg_label(selected["CNN"]["cfg"]))
    h = cnn_histories[key]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].plot(h["accuracy"], label="Training"); axes[0].plot(h["val_accuracy"], label="Early-stop set"); axes[0].set_title("Selected CNN accuracy")
    axes[1].plot(h["loss"], label="Training"); axes[1].plot(h["val_loss"], label="Early-stop set"); axes[1].set_title("Selected CNN loss")
    for a in axes: a.set_xlabel("Epoch"); a.legend()
    plt.tight_layout(); plt.show()
    gap = h["accuracy"][-1] - h["val_accuracy"][-1]
    print(f"Selected CNN ({key[1]}): {len(h['loss'])} epochs, final train-vs-early-stop accuracy gap {gap:+.4f}.")

### Validation comparison of the selected model of each family
Accuracy, macro metrics, training time, batch prediction time, **single-image latency** and model size.

In [ ]:
rows = []
for fam in FAMILIES:
    s = selected[fam]; r = s["row"]
    med, p95 = single_latency_ms(fam, s["est"], X_val)
    rows.append(dict(Family=fam, Configuration=r.Configuration, Scaling=r.Scaling, Accuracy=r.Accuracy,
                     **{"Macro Precision": r["Macro Precision"], "Macro Recall": r["Macro Recall"], "Macro F1": r["Macro F1"]},
                     **{"Train (s)": r.TrainSec, "Val batch predict (s)": r.ValPredSec,
                        "1-image latency median (ms)": med, "1-image latency p95 (ms)": p95,
                        "Model size (MB)": model_size_mb(fam, s["est"])}))
validation_results = pd.DataFrame(rows).sort_values("Macro F1", ascending=False).reset_index(drop=True)
display(validation_results.round(4))
best_family = validation_results.loc[0, "Family"]
print("Selected by validation macro-F1:", best_family, "-", validation_results.loc[0, "Configuration"])

In [ ]:
metrics = validation_results.set_index("Family")[["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]]
lo = max(0.0, metrics.values.min() - 0.02)
ax = metrics.plot(kind="bar", figsize=(9, 4)); ax.set_ylim(lo, 1.005)
ax.set(title="Validation metrics (equal 15,000-image training set)", ylabel="score"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
validation_results.set_index("Family")["Train (s)"].plot(kind="bar", ax=axes[0], title="Training time (s)", rot=0)
validation_results.set_index("Family")["1-image latency median (ms)"].plot(kind="bar", ax=axes[1], title="Single-image latency, median (ms)", rot=0)
validation_results.set_index("Family")["Model size (MB)"].plot(kind="bar", ax=axes[2], title="Model size (MB)", rot=0)
plt.tight_layout(); plt.show()

### Are the differences real? Confidence intervals and exact McNemar tests
With 6,000 validation images a gap of a few tenths of a percent can be noise. McNemar's exact test looks only at the images where two models disagree.

In [ ]:
def acc_ci(p, n): return 1.96 * np.sqrt(p * (1 - p) / n)

def mcnemar(y, pa, pb):
    a_ok, b_ok = pa == y, pb == y
    only_a, only_b = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    n = only_a + only_b
    return only_a, only_b, (binomtest(only_a, n, 0.5).pvalue if n else 1.0)

ci_rows = [dict(Family=f, Accuracy=selected[f]["row"].Accuracy, **{"95% CI (+/-)": acc_ci(selected[f]["row"].Accuracy, len(y_val))}) for f in FAMILIES]
display(pd.DataFrame(ci_rows).round(4))
pm = pd.DataFrame(index=FAMILIES, columns=FAMILIES, dtype=float)
for a in FAMILIES:
    for b in FAMILIES:
        pm.loc[a, b] = np.nan if a == b else mcnemar(y_val, selected[a]["pred"], selected[b]["pred"])[2]
print("Pairwise McNemar p-values (p < 0.05 -> the two models differ significantly):"); display(pm.round(4))

ranked = list(validation_results.Family)
if len(ranked) > 1:
    oa, ob, p_top = mcnemar(y_val, selected[ranked[0]]["pred"], selected[ranked[1]]["pred"])
    top2_note = (f"{ranked[0]} vs {ranked[1]}: {oa} images only {ranked[0]} gets right, {ob} only {ranked[1]} gets right, p = {p_top:.4f} -> "
                 + ("significant difference" if p_top < 0.05 else "NOT significant (practically tied on accuracy)"))
else:
    p_top, top2_note = 1.0, "only one model was trained"
print(top2_note)

### Per-class validation performance and confusion analysis

In [ ]:
pc = pd.DataFrame({f: precision_recall_fscore_support(y_val, selected[f]["pred"], labels=list(range(10)), zero_division=0)[2] for f in FAMILIES}).T
plt.figure(figsize=(9, 2.6)); plt.imshow(pc.values, cmap="RdYlGn", vmin=max(0.0, pc.values.min() - 0.01), vmax=1.0, aspect="auto")
plt.xticks(range(10)); plt.yticks(range(len(pc)), pc.index); plt.colorbar(label="F1")
for r in range(pc.shape[0]):
    for c in range(10): plt.text(c, r, f"{pc.values[r, c]:.3f}", ha="center", va="center", fontsize=7)
plt.title("Per-class validation F1 (columns = digits)"); plt.tight_layout(); plt.show()
weakest_val = pc.loc[best_family].sort_values().head(3)
print(f"Weakest digits for {best_family}:", {int(k): round(float(v), 4) for k, v in weakest_val.items()})

In [ ]:
best_pred_val = selected[best_family]["pred"]
cm_val = confusion_matrix(y_val, best_pred_val, labels=np.arange(10))
plt.figure(figsize=(7, 6)); plt.imshow(cm_val, cmap="Blues"); plt.colorbar()
plt.xticks(range(10)); plt.yticks(range(10)); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Validation confusion matrix — {best_family}")
for r in range(10):
    for c in range(10): plt.text(c, r, str(cm_val[r, c]), ha="center", va="center", fontsize=7)
plt.tight_layout(); plt.show()

off = cm_val.copy(); np.fill_diagonal(off, 0)
order = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1], off.shape))[0]
top_val_conf = [(int(a), int(b), int(off[a, b])) for a, b in order[:5] if off[a, b] > 0]
print("Most frequent validation confusions (actual -> predicted : count):", top_val_conf)

# Does EDA class-mean similarity predict real confusions?
pairs = [(i, j) for i in range(10) for j in range(i + 1, 10)]
rho, pval = spearmanr([sim[i, j] for i, j in pairs], [cm_val[i, j] + cm_val[j, i] for i, j in pairs])
print(f"Spearman correlation between EDA class-mean similarity and validation confusion counts: rho={rho:.2f} (p={pval:.3f})")

if top_val_conf:
    a, b, _ = top_val_conf[0]
    m = np.where((y_val == a) & (best_pred_val == b))[0][:10]
    fig, axes = plt.subplots(1, max(len(m), 1), figsize=(2 * max(len(m), 1), 2.4)); axes = np.atleast_1d(axes)
    for ax, i in zip(axes, m): ax.imshow(X_val[i].reshape(28, 28), cmap="gray"); ax.axis("off"); ax.set_title(f"True {a}\nPred {b}", fontsize=8)
    plt.suptitle("Examples of the most frequent validation confusion"); plt.tight_layout(); plt.show()

### Confidence analysis: can the model tell when it is likely wrong?
Useful in production: low-confidence predictions can be sent for human review.

In [ ]:
conf, conf_name = confidence_any(best_family, selected[best_family]["est"], X_val)
correct = best_pred_val == y_val
cdf = pd.DataFrame({"conf": conf, "correct": correct})
cdf["bin"] = pd.qcut(cdf.conf.rank(method="first"), 5, labels=["lowest 20%", "20-40%", "40-60%", "60-80%", "highest 20%"])
conf_tab = cdf.groupby("bin", observed=True).agg(images=("correct", "size"), accuracy=("correct", "mean"), mean_conf=("conf", "mean")).round(4)
display(conf_tab)
cov = []
for keep in [1.0, 0.95, 0.90, 0.80]:
    thr = np.quantile(conf, 1 - keep) if keep < 1 else -np.inf
    k = conf >= thr
    cov.append(dict(**{"Share of predictions kept": keep}, Accuracy=correct[k].mean(), **{"Errors remaining": int((~correct[k]).sum())}))
cov_df = pd.DataFrame(cov); display(cov_df.round(4))
print(f"Confidence measure: {conf_name}. Rejecting the least-confident 10% raises accuracy from {correct.mean():.4f} to {cov_df.loc[2, 'Accuracy']:.4f}.")

## Robustness: repeated validation over different 15,000-image training subsets
The selected configuration of each family is retrained on three further random equal-size subsets (same validation set, same early-stopping set). The mean ± standard deviation shows how much the ranking depends on the particular subset.

In [ ]:
rob = {f: [selected[f]["row"]["Macro F1"]] for f in FAMILIES}
for s in ROB_SEEDS:
    idx, _ = train_test_split(pool_idx, train_size=FIT_N, random_state=s, stratify=y_train[pool_idx])
    for f in FAMILIES:
        est, _ = fit_candidate(selected[f]["cfg"], X_all[idx], y_train[idx])
        rob[f].append(f1_score(y_val, predict_any(f, est, X_val), average="macro"))
    print("finished subset seed", s)
rob_df = pd.DataFrame({f: dict(mean_macroF1=np.mean(v), std=np.std(v, ddof=1) if len(v) > 1 else 0.0, min=np.min(v), max=np.max(v), runs=len(v)) for f, v in rob.items()}).T
display(rob_df.round(4))
plt.figure(figsize=(6, 3.4)); plt.errorbar(range(len(rob_df)), rob_df.mean_macroF1, yerr=rob_df["std"], fmt="o", capsize=5)
plt.xticks(range(len(rob_df)), rob_df.index); plt.ylabel("validation macro-F1"); plt.title("Stability across random training subsets"); plt.tight_layout(); plt.show()

## Secondary experiment: learning curve with equal training sizes (not part of the primary ranking)
Every family is trained on the same stratified subset sizes; the selected configuration of the primary experiment is reused. This answers "does the 15,000-image limit change the conclusion?"

In [ ]:
sizes = sorted(set(SIZES + ([len(pool_idx)] if RUN_FULL_TRAIN else [])))
lc = []
for n in sizes:
    if n > len(pool_idx): continue
    idx, _ = train_test_split(pool_idx, train_size=n, random_state=SEED, stratify=y_train[pool_idx]) if n < len(pool_idx) else (pool_idx, None)
    for f in FAMILIES:
        est, ts = fit_candidate(selected[f]["cfg"], X_all[idx], y_train[idx])
        pred = predict_any(f, est, X_val)
        lc.append(dict(Family=f, TrainSize=n, Accuracy=accuracy_score(y_val, pred), MacroF1=f1_score(y_val, pred, average="macro"), TrainSec=ts))
    print("finished size", n)
for f in FAMILIES:   # reuse the primary result at FIT_N
    r = selected[f]["row"]; lc.append(dict(Family=f, TrainSize=FIT_N, Accuracy=r.Accuracy, MacroF1=r["Macro F1"], TrainSec=r.TrainSec))
lc = pd.DataFrame(lc).sort_values(["Family", "TrainSize"]).reset_index(drop=True)
display(lc.pivot(index="TrainSize", columns="Family", values="MacroF1").round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for f in FAMILIES:
    d = lc[lc.Family == f]; axes[0].plot(d.TrainSize, d.MacroF1, marker="o", label=f); axes[1].plot(d.TrainSize, d.TrainSec, marker="o", label=f)
axes[0].set(xscale="log", xlabel="training images", ylabel="validation macro-F1", title="Learning curves"); axes[1].set(xscale="log", yscale="log", xlabel="training images", ylabel="training seconds", title="Training cost")
axes[0].legend(); plt.tight_layout(); plt.show()

# Final evaluation on the official test set (selected model only, once)
The model family and hyper-parameters were chosen on validation data. The cell below evaluates **only** that model; the result is not used to change anything.

In [ ]:
sel = selected[best_family]
test_pred, test_secs = timed_predict(best_family, sel["est"], X_test_flat)
test_metrics = score_predictions(y_test, test_pred)
test_ci = acc_ci(test_metrics["Accuracy"], len(y_test))
display(pd.DataFrame([test_metrics]).round(4))
print(f"Test accuracy {test_metrics['Accuracy']:.4f} +/- {test_ci:.4f} (95% CI); {int((test_pred != y_test).sum())} of {len(y_test)} images misclassified; "
      f"batch prediction time {test_secs:.2f}s")
rep = pd.DataFrame(classification_report(y_test, test_pred, digits=4, zero_division=0, output_dict=True)).T
display(rep.round(4))

In [ ]:
cm_te = confusion_matrix(y_test, test_pred, labels=np.arange(10))
plt.figure(figsize=(7, 6)); plt.imshow(cm_te, cmap="Blues"); plt.colorbar()
plt.xticks(range(10)); plt.yticks(range(10)); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Official test confusion matrix — {best_family}")
for r in range(10):
    for c in range(10): plt.text(c, r, str(cm_te[r, c]), ha="center", va="center", fontsize=7)
plt.tight_layout(); plt.show()

off_te = cm_te.copy(); np.fill_diagonal(off_te, 0)
order = np.dstack(np.unravel_index(np.argsort(off_te.ravel())[::-1], off_te.shape))[0]
top_test_conf = [(int(a), int(b), int(off_te[a, b])) for a, b in order[:5] if off_te[a, b] > 0]
print("Most frequent test confusions (actual -> predicted : count):", top_test_conf)

t_conf, _ = confidence_any(best_family, sel["est"], X_test_flat)
wrong = np.where(test_pred != y_test)[0]
show = wrong[np.argsort(t_conf[wrong])[::-1]][:10]     # most confidently wrong first
if len(show):
    fig, axes = plt.subplots(1, len(show), figsize=(2 * len(show), 2.7)); axes = np.atleast_1d(axes)
    for ax, i in zip(axes, show):
        ax.imshow(x_test[i], cmap="gray"); ax.axis("off"); ax.set_title(f"True {y_test[i]}  Pred {test_pred[i]}\nconf {t_conf[i]:.2f}", fontsize=7)
    plt.suptitle("Most confidently wrong test predictions (qualitative error analysis)"); plt.tight_layout(); plt.show()

## Save the selected model and an inference helper

In [ ]:
import joblib
if best_family == "CNN":
    sel["est"].save("handwritten_digits_cnn.keras"); saved = "handwritten_digits_cnn.keras"
else:
    saved = f"handwritten_digits_{best_family.lower()}.joblib"; joblib.dump(sel["est"], saved)
print("Saved:", saved)

def predict_digit(image, family=best_family, est=sel["est"]):
    """Classify one 28x28 image (white digit on black background, like MNIST). Real photos must be inverted,
    cropped to the digit, centred and resized first."""
    a = np.asarray(image, dtype="float32").reshape(1, -1)
    a = a / 255.0 if a.max() > 1.0 else a
    return int(predict_any(family, est, a)[0])

demo = [predict_digit(x_test[i]) == y_test[i] for i in range(20)]
print("Inference helper check on 20 test images:", sum(demo), "/ 20 correct")

# Model Comparison Report (generated from this run)

In [ ]:
def pct(x): return f"{100 * x:.2f}%"
vr = validation_results.set_index("Family")
top = vr.loc[best_family]
second = ranked[1] if len(ranked) > 1 else None
fast_fam = vr["1-image latency median (ms)"].idxmin(); small_fam = vr["Model size (MB)"].idxmin()

lc_piv = lc.pivot(index="TrainSize", columns="Family", values="MacroF1")
lc_gain = {f: (lc_piv[f].iloc[0], lc_piv[f].iloc[-1]) for f in FAMILIES}
lc_txt = "; ".join(f"{f}: {a:.3f} at {lc_piv.index[0]:,} images -> {b:.3f} at {lc_piv.index[-1]:,}" for f, (a, b) in lc_gain.items())
rob_txt = "; ".join(f"{f} {rob_df.loc[f, 'mean_macroF1']:.4f} +/- {rob_df.loc[f, 'std']:.4f}" for f in FAMILIES)

alt = [f for f in FAMILIES if f != best_family and vr.loc[best_family, "Macro F1"] - vr.loc[f, "Macro F1"] <= 0.005
       and vr.loc[f, "1-image latency median (ms)"] * 3 <= top["1-image latency median (ms)"]]
if second and p_top >= 0.05:
    rec_extra = f" The gap to {second} is not statistically significant (McNemar p = {p_top:.3f}), so cost and latency should weigh in the deployment decision."
elif second:
    rec_extra = f" The advantage over {second} is statistically significant (McNemar p = {p_top:.4f})."
else:
    rec_extra = ""
if alt:
    rec_extra += f" {', '.join(alt)} is within 0.5 points of macro-F1 and at least 3x faster per image, making it a sensible lightweight fallback."

report = f"""
## Method
{len(FAMILIES)} model families ({', '.join(FAMILIES)}{'' if HAS_TF else '; CNN skipped because TensorFlow is missing'}) were trained on the **same stratified {len(y_fit):,}-image subset** and compared on the **same {len(y_val):,}-image validation set**. Normalisation-only vs added standardisation was decided by ablation ({', '.join(f + ': ' + ('standardised' if v == 'standard' else '[0,1] only') for f, v in best_scaling.items() if f != 'CNN')}); hyper-parameters came from the grids above; MLP/CNN early-stopped on a separate {len(y_es):,}-image set. The official test set was used once, for the validation-selected model.

## Validation ranking
{df_to_md(validation_results[['Family','Configuration','Accuracy','Macro F1','Train (s)','1-image latency median (ms)','Model size (MB)']])}

## Selected model
**{best_family}** ({top.Configuration}; scaling: {top.Scaling}) — validation accuracy {pct(top.Accuracy)} (±{100*acc_ci(top.Accuracy, len(y_val)):.2f} pts, 95% CI), macro-F1 {top['Macro F1']:.4f}.
**Official test:** accuracy {pct(test_metrics['Accuracy'])} (±{100*test_ci:.2f} pts), macro-precision {test_metrics['Macro Precision']:.4f}, macro-recall {test_metrics['Macro Recall']:.4f}, macro-F1 {test_metrics['Macro F1']:.4f}; {int((test_pred != y_test).sum())} of {len(y_test):,} test images misclassified.

## Evidence behind the choice
- **Significance:** {top2_note}.
- **Stability:** mean validation macro-F1 over {len(ROB_SEEDS)+1} random training subsets — {rob_txt}.
- **Data size:** {lc_txt}.
- **Error patterns:** most frequent test confusions (actual→predicted, count): {top_test_conf}. Weakest validation digits for {best_family}: {[int(k) for k in weakest_val.index]}. EDA class-mean similarity vs real confusions: Spearman rho = {rho:.2f}.
- **Confidence:** rejecting the least-confident 10% of predictions lifts validation accuracy from {pct(correct.mean())} to {pct(cov_df.loc[2, 'Accuracy'])}, so low-confidence cases can be routed to review.
- **Cost:** fastest single-image model is {fast_fam} ({vr.loc[fast_fam, '1-image latency median (ms)']:.2f} ms median); smallest is {small_fam} ({vr.loc[small_fam, 'Model size (MB)']:.2f} MB). Selected {best_family}: {top['1-image latency median (ms)']:.2f} ms/image, {top['Model size (MB)']:.2f} MB, {top['Train (s)']:.1f} s to train.

## Recommendation for production
Deploy **{best_family}** (`{saved}`): it has the best validation macro-F1 and its held-out test accuracy is {pct(test_metrics['Accuracy'])}.{rec_extra} Before real-world use, validate on handwriting from the target population and device, because MNIST digits are clean, centred and size-normalised.

## Limitations
Primary comparison uses {len(y_fit):,} of {len(pool_idx)+len(val_idx)+len(es_idx):,} training images (the learning curve quantifies the effect); grids are small, not exhaustive cross-validation; timings depend on this machine; confidence scores are not calibrated probabilities; MNIST-specific.
"""
display(Markdown(report))

# Challenges Faced and Techniques Used
| Challenge | Technique | Why it was used |
|---|---|---|
| Pixels span 0–255 | Converted to float32 in [0, 1] | Common input scale; stable neural-network optimisation |
| Classical models need vectors, CNN needs spatial structure | Flattened to 784 features for KNN/SVM/MLP; kept 28×28×1 for CNN | Matches each model's input format and preserves local structure for convolutions |
| Unsure whether extra scaling helps | Ablation: [0,1] only vs + `StandardScaler`, decided on validation macro-F1 | Evidence instead of assumption; scaler is fitted on training data only, inside a `Pipeline` |
| Unequal training sizes make comparisons unfair | Same stratified 15,000-image subset and same validation set for every model | Differences are due to the algorithm, not the data volume |
| Early stopping on the comparison set would leak into model selection | Separate early-stopping set for MLP and CNN | The validation set stays untouched for comparison; test set used once |
| Narrow tuning could miss better settings | Wider grids for KNN, SVM (C × gamma), MLP (width/depth × L2); three CNN designs incl. augmentation | Gives each family a fair chance without exhaustive search |
| One split can mislead | Repeated random subsets (mean ± std), McNemar tests, confidence intervals | Shows whether a ranking is real or noise |
| 15,000 images may not reflect the full data | Separate learning-curve experiment over equal sizes | Tests whether conclusions change with more data without contaminating the primary ranking |
| Visually similar or broken digits | Class-similarity map, connected components, confusion matrices, most-confidently-wrong images | Explains where and why errors occur |
| Accuracy alone hides class and cost issues | Macro metrics, per-class F1, single-image latency, model size, confidence analysis | Production choice needs accuracy and cost, and a way to flag doubtful predictions |
| Reproducibility | Fixed seeds, logged library versions, saved model, outputs kept in the notebook | Anyone can re-run and verify |

# Conclusion
The generated report above names the selected model, its hyper-parameters, validation and test metrics, the main confusion patterns, and the cost trade-offs. The cell below prints a short summary suitable for the project submission form.

In [ ]:
summary = (f"Project: PRCP-1002 Handwritten Digits (MNIST). {len(FAMILIES)} models ({', '.join(FAMILIES)}) were compared on an identical "
           f"{len(y_fit):,}-image training subset and a {len(y_val):,}-image validation set; MLP/CNN early-stopped on a separate set; "
           f"the official test set was used once. Selected model by validation macro-F1: {best_family} ({top.Configuration}), "
           f"validation accuracy {pct(top.Accuracy)}, macro-F1 {top['Macro F1']:.4f}; official test accuracy {pct(test_metrics['Accuracy'])}, "
           f"macro-F1 {test_metrics['Macro F1']:.4f}. {top2_note}. Most common test confusion: "
           f"{top_test_conf[0][0]} predicted as {top_test_conf[0][1]} ({top_test_conf[0][2]} image(s))." if top_test_conf else "No test errors.")
print(summary)